# 🤖 Lekcja 4: Nasz pierwszy model Machine Learningu! (`train_test_split`, Las Losowy i analiza błędów)

Nadszedł wielki moment! Poznaliśmy już dane, zobaczyliśmy wykresy i nauczyliśmy się łatać dziury `NaN`. Dziś zbudujemy **nasz pierwszy prawdziwy model Machine Learningu**, który samodzielnie wyceni domy!

Zanim uruchomimy kod, wyjaśnijmy sobie po ludzku dwa fundamentalne pojęcia, bez których nie istnieje żadne poważne Machine Learning:

---

### 1. 🌳 Jak myśli Drzewo Decyzyjne (*Decision Tree*), a jak Las Losowy (*Random Forest*)?

* **Pojedyncze Drzewo Decyzyjne (*Decision Tree*)** działa dokładnie jak **gra w „20 pytań”** (albo schemat blokowy lekarza diagnozującego pacjenta).  
  Patrzy na dom i zadaje serię prostych pytań:
  1. *„Czy ocena jakości `OverallQual` jest większa niż 7?”* $\rightarrow$ **TAK**
  2. *„Czy powierzchnia `GrLivArea` jest większa niż 2000 stóp?”* $\rightarrow$ **TAK**
  3. *„Czy garaż `GarageCars` mieści 3 auta?”* $\rightarrow$ **NIE**  
  Na samym dole drzewa (w tzw. **liściu**) znajduje się np. 12 historycznych domów, które odpowiedziały na pytania w ten sam sposób. Drzewo liczy ich średnią cenę (np. `285 000 $`) i mówi: *„Moja wycena to 285 000 \$!”*.

* **Problem z pojedynczym drzewem:** Jeden rzeczoznawca bywa kapryśny i łatwo uczy się danych „na pamięć”.
* **Rozwiązanie — Las Losowy (`RandomForestRegressor`):** Zamiast pytać jednego rzeczoznawcę, powołujemy **Radę 100 niezależnych rzeczoznawców (100 różnych drzew decyzyjnych)**!  
  Każde z tych 100 drzew uczy się na nieco innym, losowo wybranym podzbiorze domów. Kiedy przychodzi nowy dom do wyceny:
  * Drzewo nr 1 mówi: `270 000 $`
  * Drzewo nr 2 mówi: `295 000 $`
  * ...i tak dalej dla wszystkich 100 drzew.
  * Na końcu komputer wyciąga **średnią arytmetyczną ze wszystkich 100 głosów**! Dzięki tzw. *mądrości tłumu* błędy pojedynczych drzew znoszą się nawzajem.

---

### 2. 🔐 Po co dzielimy dane na dwie części (`train_test_split`)? (Analogia próbnej matury)

Pamiętasz z Lekcji 1, że w pliku egzaminacyjnym `../data/test.csv` **nie ma cen `SalePrice`** (zna je tylko serwis Kaggle)?  
Skąd więc mamy wiedzieć u siebie na komputerze, czy nasz model wycenia domy dobrze, czy plecie bzdury?

* **Błąd początkującego:** Wytrenować model na wszystkich 1460 domach z pliku `../data/train.csv`, a potem kazać mu wycenić... te same 1460 domów!  
  Dlaczego to oszustwo? Wyobraź sobie nauczyciela, który dzień przed maturą daje uczniowi arkusz z zadaniami i odpowiedziami, a następnego dnia na egzaminie daje mu **dokładnie te same zadania**. Uczeń dostanie 100%, bo wykuł odpowiedzi na blachę (w ML nazywamy to **przeuczeniem / *Overfitting***), ale gdy dostanie nowe zadanie, polegnie!

* **Jak robią to profesjonaliści (`train_test_split`):**  
  Bierzemy nasz plik `../data/train.csv` (1460 domów ze znanymi cenami) i **przecinamy go na dwie części**:
  1. 📚 **Zbiór treningowy (`80%` danych = 1168 domów)** — dajemy go modelowi do nauki.
  2. 🔒 **Zbiór walidacyjny (`20%` danych = 292 domy)** — **zamykamy w sejfie!** Model nie widzi tych 292 domów podczas nauki. Dopiero gdy skończy się uczyć, wyjmujemy z sejfu cechy tych 292 domów (bez cen!), każemy modelowi zgadnąć ich ceny, a potem porównujemy jego wyceny z prawdziwymi cenami z sejfu!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Wyciągamy narzędzia z biblioteki scikit-learn (sklearn) — głównej skrzynki do Machine Learningu
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1. Wczytujemy dane z folderu ../data/
train = pd.read_csv('../data/train.csv')

# 2. Wybieramy na start tylko 5 prostych, liczbowych cech (nie mają żadnych braków NaN w train.csv!)
cechy = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'YearBuilt']

X = train[cechy]          # X (wielka litera) = tabela z cechami domów (pytania egzaminacyjne)
y = train['SalePrice']    # y (mała litera) = kolumna z cenami (klucz odpowiedzi)

# 3. Przecinamy dane: 80% do nauki (train), 20% do sejfu na sprawdzian (val)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Domy do nauki (X_train): {X_train.shape[0]} domów, {X_train.shape[1]} cech')
print(f'Domy w sejfie na egzamin (X_val): {X_val.shape[0]} domów, {X_val.shape[1]} cech\n')

# 4. Tworzymy nasz pierwszy model: Las Losowy złożony ze 100 drzew decyzyjnych
model = RandomForestRegressor(n_estimators=100, random_state=42)

# 5. NAUKA MODELU (.fit) — praktykant uczy się tylko na 1168 domach treningowych!
model.fit(X_train, y_train)

# 6. EGZAMIN (.predict) — dajemy modelowi same cechy 292 domów z sejfu (X_val) i prosimy o wycenę!
predykcje = model.predict(X_val)

# 7. SPRAWDZENIE WYNIKÓW — o ile dolarów średnio pomylił się nasz model na 292 domach z sejfu?
sredni_blad = mean_absolute_error(y_val, predykcje)
print(f'🎯 Sukces! Nasz pierwszy model działa!')
print(f'Średni błąd wyceny (MAE) na 292 nieznanych domach wynosi: {sredni_blad:,.0f} $')

### 🔬 Szczegółowa rozbiórka matematyczna i programistyczna (Co dokładnie wydarzyło się w kodzie?)

Nasz model nauczył się wyceniać domy na podstawie zaledwie **5 liczb** (spośród 80 dostępnych kolumn!) i na 292 zupełnie obcych domach myli się średnio o **`19 180 $`** (przy średniej cenie domu ok. 180 000 \$ to błąd rzędu ~10%). Rozłóżmy każdą linijkę i każde pojęcie matematyczne na czynniki pierwsze:

---

#### 1. Dlaczego `X` piszemy WIELKĄ literą, a `y` MAŁĄ literą?
W każdym kursie i książce o Machine Learningu na świecie zobaczysz zmienną `X` oraz `y`. To nie przypadek — to konwencja prosto z matematyki (algebry liniowej):
* **`X` (Wielka litera)** oznacza **MACIERZ (tabelę dwuwymiarową)**, która ma wiele wierszy i wiele kolumn (u nas 1460 wierszy $\times$ 5 kolumn z cechami domu).
* **`y` (Mała litera)** oznacza **WEKTOR (pojedynczą kolumnę)** z jedną wartością dla każdego domu — czyli naszą cenę docelową (`SalePrice`), którą chcemy przewidzieć.
* Cały Machine Learning to po prostu szukanie matematycznej funkcji $y = f(X)$, która z tabeli cech $X$ obliczy wynik $y$!

---

#### 2. Jak działa `X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)`?

Funkcja `train_test_split` bierze naszą tabelę `X` oraz kolumnę cen `y`, tasuje wiersze jak talię kart i przecina je w poziomie na cztery kawałki:

| Nazwa zmiennej | Co w niej siedzi? | Wymiary (`.shape`) | Rola w naszym eksperymencie |
| :--- | :--- | :--- | :--- |
| **`X_train`** | 5 cech dla 80% domów | `(1168, 5)` | **Podręcznik (pytania)**, z którego model uczy się zależności. |
| **`y_train`** | Prawdziwe ceny dla tych 80% domów | `(1168,)` | **Podręcznik (odpowiedzi)**, dzięki którym model wie, ile kosztowały domy z `X_train`. |
| **`X_val`** | 5 cech dla pozostałych 20% domów | `(292, 5)` | **Arkusz egzaminacyjny w sejfie** (`test_size=0.2` oznacza 20% = 0.2). Podajemy go do `.predict(X_val)`. |
| **`y_val`** | Prawdziwe ceny dla tych 20% domów | `(292,)` | **Klucz odpowiedzi egzaminatora w sejfie**. Model NIGDY go nie widzi! Używamy go tylko my na końcu, by policzyć błąd. |

> 💡 **Co oznacza `random_state=42`? (Ziarno losowości)**  
> Zanim `train_test_split` odetnie 20% domów, **losowo tasuje wiersze** (żeby np. wszystkie domy z końca tabeli nie trafiły razem do sejfu).  
> Gdybyśmy nie podali `random_state`, przy każdym kliknięciu `Shift + Enter` komputer tasowałby domy inaczej i do sejfu trafiałyby inne 292 domy, a nasz wynik skakałby za każdym uruchomieniem!  
> Wpisując **`random_state=42`**, zamrażamy generator losowy w jednym konkretnym ułożeniu kart — dzięki temu zawsze mamy w sejfie **dokładnie te same 292 domy**!

---

#### 3. Trzy święte kroki w `scikit-learn`: Stwórz $\rightarrow$ `.fit()` $\rightarrow$ `.predict()`
Niezależnie od tego, czy budujesz proste Drzewo Decyzyjne, Las Losowy, czy zaawansowany model za milion dolarów, w bibliotece `sklearn` kod wygląda **zawsze tak samo** (to tylko 3 linijki!):
1. **`model = RandomForestRegressor(n_estimators=100, random_state=42)`** — tworzymy „pusty mózg” modelu. Parametr `n_estimators=100` mówi: *„zbuduj las złożony ze 100 drzew decyzyjnych”* (`estimator` to po angielsku właśnie model szacujący / rzeczoznawca).
2. **`model.fit(X_train, y_train)`** — słowo `fit` oznacza **„dopasuj się / naucz się”**. Tu podajemy **i pytania (`X_train`), i odpowiedzi (`y_train`)**. Komputer buduje w pamięci 100 drzew decyzyjnych.
3. **`predykcje = model.predict(X_val)`** — słowo `predict` oznacza **„przewiduj / zgaduj”**. Zauważ, że w nawiasie podajemy **TYLKO `X_val` (bez `y_val`!)**. Model zwraca listę 292 wyliczonych przez siebie cen.

---

#### 4. Matematyka miary błędu: Czym jest MAE (`mean_absolute_error` = 19 180 \$)?

Skąd wzięła się liczba **`19 180 $`**? To **Średni Błąd Bezwzględny** (po angielsku ***Mean Absolute Error*** — w skrócie **MAE**).  
Zajrzyjmy pod maskę, jak komputer liczy to w 3 prostych krokach:

$$\text{MAE} = \frac{1}{292} \sum_{i=1}^{292} | \text{Cena Prawdziwa}_i - \text{Cena Przewidziana}_i |$$

1. **Krok 1 (Różnica):** Dla każdego z 292 domów z sejfu odejmujemy wycenę modelu od prawdziwej ceny.
2. **Krok 2 (Wartość bezwzględna $|\dots|$, czyli `Absolute`):** Dlaczego ucinamy znak minus?  
   Wyobraź sobie, że na pierwszym domu model pomylił się o **`+50 000 $`** (zawyżył), a na drugim domu pomylił się o **`-50 000 $`** (zaniżył). Gdybyśmy po prostu dodali te błędy do siebie: $+50\,000 + (-50\,000) = 0\ \$$, wyszłoby nam, że model ma średni błąd **ZERO dolarów**, choć na obu domach pomylił się o 50 tysięcy!  
   Dlatego pionowe kreski $| -50\,000 | = +50\,000$ zamieniają każdy błąd na liczbę dodatnią.
3. **Krok 3 (Średnia, czyli `Mean`):** Dodajemy wszystkie 292 dodatnie pomyłki do siebie i dzielimy przez 292. Wynik **`19 180 $`** mówi po ludzku: *„Nasz model myli się na jednym domu średnio o 19 180 dolarów w górę lub w dół”*.

In [ ]:
# 1. Tworzymy tabelę porównawczą dla wszystkich 292 domów z naszego sejfu (X_val)
wyniki = X_val.copy()
wyniki['Id'] = train.loc[X_val.index, 'Id']
wyniki['Prawdziwa_Cena'] = y_val
wyniki['Przewidziana_Cena'] = np.round(predykcje, 0)
wyniki['Blad_Dolarowy'] = np.round(predykcje - y_val, 0)         # + oznacza zawyżenie, - oznacza zaniżenie
wyniki['Blad_Bezwzgledny'] = np.round(np.abs(predykcje - y_val), 0)

# Dodajmy do podglądu kilka ukrytych kolumn, których nasz model NIE WIDZIAŁ podczas nauki:
ukryte_kolumny = ['Neighborhood', 'LotArea', 'YearRemodAdd', 'OverallCond', 'BsmtFinSF1', 'BsmtFinType1', 'SaleCondition']
for kol in ukryte_kolumny:
    wyniki[kol] = train.loc[X_val.index, kol]

# 2. Wybieramy 5 domów, na których model pomylił się NAJBARDZIEJ
top5_bledow = wyniki.sort_values(by='Blad_Bezwzgledny', ascending=False).head(5)

# 3. Rysujemy DWA wykresy obok siebie
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# WYKRES LEWY (ax1): Prawdziwa cena vs Przewidziana cena dla wszystkich 292 domów
ax1.scatter(wyniki['Prawdziwa_Cena'], wyniki['Przewidziana_Cena'], alpha=0.5, color='royalblue', label='Domy ze zbioru walidacyjnego (292)')
ax1.scatter(top5_bledow['Prawdziwa_Cena'], top5_bledow['Przewidziana_Cena'], color='red', s=90, zorder=5, label='5 największych pomyłek modelu')

# Rysujemy czerwoną przerywaną linię idealnych trafień (gdzie Przewidziana == Prawdziwa)
ax1.plot([0, 760000], [0, 760000], color='darkred', linestyle='--', linewidth=2, label='Linia idealnej wyceny (0$ błędu)')

# Podpisujemy 5 czerwonych kropek ich numerami Id
for _, wiersz in top5_bledow.iterrows():
    ax1.annotate(f"Id={int(wiersz['Id'])}", 
                 (wiersz['Prawdziwa_Cena'], wiersz['Przewidziana_Cena']),
                 textcoords='offset points', xytext=(6, -4), fontsize=9, fontweight='bold', color='darkred')

ax1.set_title('1. Prawdziwa cena vs Cena przewidziana przez model')
ax1.set_xlabel('Prawdziwa cena sprzedaży — y_val ($)')
ax1.set_ylabel('Cena przewidziana przez model — predykcje ($)')
ax1.legend()
ax1.grid(True, linestyle=':', alpha=0.6)

# WYKRES PRAWY (ax2): Porównanie słupkowe dla 5 największych pomyłek
etykiety_domow = [f"Id {int(i)}" for i in top5_bledow['Id']]
pozycje = np.arange(len(etykiety_domow))
szerokosc = 0.36

ax2.bar(pozycje - szerokosc/2, top5_bledow['Prawdziwa_Cena'], width=szerokosc, color='royalblue', edgecolor='black', label='Prawdziwa cena ($)')
ax2.bar(pozycje + szerokosc/2, top5_bledow['Przewidziana_Cena'], width=szerokosc, color='coral', edgecolor='black', label='Wycena modelu ($)')

ax2.set_title('2. Top 5 największych pomyłek modelu (Prawda vs Model)')
ax2.set_xticks(pozycje)
ax2.set_xticklabels(etykiety_domow)
ax2.set_ylabel('Cena w dolarach ($)')
ax2.legend()
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

# 4. Wyświetlamy tabelę ze szczegółami tych 5 domów
kolumny_do_pokazania = [
    'Id', 'Prawdziwa_Cena', 'Przewidziana_Cena', 'Blad_Dolarowy',
    'OverallQual', 'GrLivArea', 'TotalBsmtSF', 'YearBuilt',
    'YearRemodAdd', 'LotArea', 'BsmtFinSF1', 'BsmtFinType1', 'Neighborhood', 'SaleCondition'
]
top5_bledow[kolumny_do_pokazania]

### 🕵️‍♂️ Jak czytać ten wykres i dlaczego model pomylił się na tych 5 domach?

#### 1. Jak czytać lewy wykres (Prawdziwa cena vs Przewidziana cena)?
To najważniejszy wykres diagnostyczny w całym Machine Learningu:
* **Czerwona przerywana przekątna** to **Linia Idealnego Rzeczoznawcy**. Gdyby dla domu wartego `200 000 $` (oś X) model przewidział dokładnie `200 000 $` (oś Y), kropka wylądowałaby idealnie na czerwonej linii!
* **Kropki POD czerwoną linią** to domy, których wartość model **ZANIŻYŁ** (przewidział mniej, niż wynosiła prawdziwa cena).
* **Kropki NAD czerwoną linią** to domy, których wartość model **ZAWYŻYŁ** (przewidział więcej, niż dom był wart).
* Zobacz, że w przedziale **100 000 – 300 000 \$** niebieskie kropki pięknie tulą się do czerwonej linii! Ale przy drogich i nietypowych domach pojawiają się nasze **czerwone kropki (Top 5 pomyłek)**.

---

#### 2. Śledztwo detektywistyczne: Dlaczego nasz 5-cechowy model poległ na tych 5 konkretnych domach?

Pamiętaj, że daliśmy naszemu modelowi tylko **5 okularów (5 kolumn)**: `OverallQual`, `GrLivArea`, `GarageCars`, `TotalBsmtSF` oraz `YearBuilt`. Dla pozostałych 75 kolumn nasz model był **całkowicie ślepy**! Spójrzmy do tabeli powyżej, co ukrywało się w tych 5 domach:

| Miejsce | Dom (`Id`) | Prawdziwa cena vs Model (Błąd) | Co widział nasz model (5 cech)? | Czego model NIE WIDZIAŁ (Dlaczego się pomylił)? |
| :--- | :--- | :--- | :--- | :--- |
| **1.** | **`Id = 899`** | Prawda: **611 657 \$**<br>Model: **473 865 \$**<br>*(Zaniżył o -137 792 \$)* | Widział duży, nowy dom (`GrLivArea = 2364`, `TotalBsmtSF = 2330`, `OverallQual = 9`), więc wycenił go na solidne 474 tys. \$. | Model widział rozmiar piwnicy (`2330`), ale **nie widział kolumny `BsmtFinSF1 = 2188` oraz `BsmtFinType1 = 'GLQ'`**! W tym domu niemal cała ogromna piwnica (ponad 200 $\text{m}^2$!) została wykończona na **luksusowy poziom mieszkalny** z wyjściem na ogród, w najdroższej dzielnicy `NridgHt`! |
| **2.** | **`Id = 582`** | Prawda: **253 293 \$**<br>Model: **377 721 \$**<br>*(Zawyżył o +124 428 \$)* | Widział nowiutki dom z 2008 roku (`OverallQual = 8`, `GrLivArea = 2042`, wielka piwnica `TotalBsmtSF = 2042`, garaż na 3 auta). | Dokładne przeciwieństwo domu wyżej! Model myślał, że wielka piwnica (`2042`) to świetny atut, ale nie widział, że **`BsmtFinSF1 = 0` oraz `BsmtFinType1 = 'Unf'` (cała piwnica to surowy, niewykończony beton!)**, a `SaleCondition = 'Partial'` (dom sprzedano w nieukończonym stanie!). |
| **3.** | **`Id = 692`** | Prawda: **755 000 \$**<br>Model: **634 135 \$**<br>*(Zaniżył o -120 865 \$)* | Widział absolutnego giganta: `OverallQual = 10`, `GrLivArea = 4316` stóp ($>400\text{ m}^2$). | **To najdroższy dom w całej bazie (755 000 \$)!** Tutaj wychodzi na jaw **ważna cecha matematyczna Lasu Losowego**: ponieważ każde drzewo podaje średnią z domów, które widziało podczas nauki w `X_train`, **Las Losowy NIGDY nie potrafi przewidzieć liczby wyższej niż najdroższe domy ze zbioru treningowego**! Ponadto nie widział ogromnej działki (`LotArea = 21 535`). |
| **4.** | **`Id = 219`** | Prawda: **311 500 \$**<br>Model: **191 460 \$**<br>*(Zaniżył o -120 040 \$)* | Zobaczył **`YearBuilt = 1939`** (stary, przedwojenny dom!) oraz małą piwnicę (`798` stóp), więc mocno obciął wycenę do 191 tys. \$. | Model nie widział kolumny **`YearRemodAdd = 2006`** (w 2006 roku dom przeszedł generalną przebudowę!), kolumny **`OverallCond = 9`** (stan techniczny 9/10!) oraz prestiżowej dzielnicy `'Crawfor'`. |
| **5.** | **`Id = 272`** | Prawda: **241 500 \$**<br>Model: **148 314 \$**<br>*(Zaniżył o -93 186 \$)* | Zobaczył malutki dom (`GrLivArea = 1363` stopy $\approx 126\text{ m}^2$) z `1954` roku, więc wycenił go jak skromny domek na 148 tys. \$. | Model nie widział kolumny **`LotArea = 39 104` stopy kwadratowe**! Kupujący zapłacił 241 500 \$ nie za sam mały budynek, ale za **gigantyczną, prawie 40-arową działkę w leśnej dzielnicy `ClearCr`** (4 razy większą od zwykłej działki!) po remoncie w 2005 roku! |

---

💡 **Wielka lekcja Data Science:** Zobacz, jak analiza błędów otwiera oczy! Zamiast zgadywać po omacku, czego brakuje naszemu modelowi, spojrzeliśmy na 5 największych pomyłek i od razu odkryliśmy, jakie kolejne kolumny musimy mu dać: **rok remontu (`YearRemodAdd`)**, **powierzchnię działki (`LotArea`)**, **wykończoną powierzchnię piwnicy (`BsmtFinSF1`)** oraz **dzielnicę (`Neighborhood`)**!

In [ ]:
# 🎯 ĆWICZENIE 1: Pojedynek! 1 Rzeczoznawca (1 drzewo) kontra Rada 100 Rzeczoznawców (100 drzew)
# Sprawdźmy na własne oczy, o ile gorzej poradzi sobie pojedyncze drzewo decyzyjne (`n_estimators=1`)
# w porównaniu do naszego lasu 100 drzew (który miał błąd MAE = 19 180 $).
#
# 1. Stwórz zmienną `jedno_drzewo = RandomForestRegressor(n_estimators=1, random_state=42)`
# 2. Naucz `jedno_drzewo` na danych treningowych za pomocą metody .fit(X_train, y_train)
# 3. Wygeneruj predykcje na X_val metodą .predict(X_val) i oblicz błąd mean_absolute_error(y_val, ...)

# Miejsce na Twój kod:


In [ ]:
# 🎯 ĆWICZENIE 2: Dajemy modelowi nowe okulary! (Naprawiamy błędy z naszego śledztwa)
# W śledztwie odkryliśmy, że model mylił się, bo nie znał roku remontu ('YearRemodAdd'),
# wielkości działki ('LotArea') oraz wykończonej części piwnicy ('BsmtFinSF1').
# Dodajmy te 3 kolumny (dając mu łącznie 8 cech zamiast 5) i zobaczmy, czy błąd MAE spadnie poniżej 19 180 $!

nowe_cechy = [
    'OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'YearBuilt',
    'YearRemodAdd', 'LotArea', 'BsmtFinSF1'
]

X_nowe = train[nowe_cechy]
X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(X_nowe, y, test_size=0.2, random_state=42)

# 1. Stwórz model `lepszy_model = RandomForestRegressor(n_estimators=100, random_state=42)`
# 2. Wytrenuj `lepszy_model` metodą .fit() na (X_train_2, y_train_2)
# 3. Wygeneruj predykcje na X_val_2 metodą .predict() i oblicz nowy mean_absolute_error(y_val_2, ...)

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Podział `train_test_split(..., test_size=0.2, random_state=42)`:** Aby uczciwie sprawdzić model przed prawdziwym egzaminem, odkładamy 20% domów (292 domy) do „sejfu walidacyjnego” (`X_val`, `y_val`), a model uczymy wyłącznie na pozostałych 80% domów (`X_train`, `y_train`).
2. **Las Losowy (`RandomForestRegressor`) i 3 kroki `sklearn`:** Zamiast polegać na jednym drzewie decyzyjnym (błąd ~28 105 \$), łączymy głosy 100 niezależnych drzew (błąd spada do 19 180 \$), używając uniwersalnego schematu: stworzenie modelu $\rightarrow$ `.fit(X_train, y_train)` $\rightarrow$ `.predict(X_val)`.
3. **Analiza największych błędów (MAE):** Badając 5 domów, na których nasz 5-cechowy model pomylił się najbardziej, odkryliśmy, jakich informacji mu brakowało (wykończenia piwnicy `BsmtFinSF1`, roku remontu `YearRemodAdd` oraz powierzchni działki `LotArea`) — ich dodanie obniża błąd do 18 380 \$.